In [ ]:
# MultiCodeAttack on Qwen2.5-3B-Instruct (Kaggle, GPU T4 x2, Internet ON)
TASK = "code"  # "template-only", "text", or "code"
NUM_SAMPLES = 10     # sentence groups; 0 = all 520

In [ ]:
import os
repo = "https://github.com/brainardphilemon/MultiCodeAttack.git"
!rm -rf /kaggle/working/MultiCodeAttack
!git clone -q {repo} /kaggle/working/MultiCodeAttack
%cd /kaggle/working/MultiCodeAttack
!pip -q install vllm openai nltk pandas tqdm python-dotenv
# vllm pulls a CUDA-13 torch; Kaggle's preinstalled torchaudio is CUDA 12.8 and
# crashes vllm on import (transformers imports it). vllm doesn't need it.
!pip -q uninstall -y torchaudio
!nvidia-smi --query-gpu=name,driver_version --format=csv
import subprocess as _sp
print(_sp.run(["python", "-c", "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
              capture_output=True, text=True).stdout)
import nltk
for p in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(p, quiet=True)

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-3B-Instruct-AWQ",
    "--served-model-name", "Qwen2.5-3B-Instruct",
    "--quantization", "awq", "--port", "8000",
    "--max-model-len", "4096", "--gpu-memory-utilization", "0.90",
], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    if server.poll() is not None:
        log.flush()
        print(open("/kaggle/working/vllm.log").read()[-4000:])
        raise RuntimeError(f"vLLM exited with code {server.returncode}")
    try:
        if requests.get("http://localhost:8000/health").status_code == 200:
            print("vLLM server is up"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open("/kaggle/working/vllm.log").read()[-4000:])
    raise RuntimeError("vLLM did not come up; see /kaggle/working/vllm.log")

In [ ]:
%cd /kaggle/working/MultiCodeAttack
os.environ.update(PRIMARY_MODEL="Qwen2.5-3B-Instruct",
                  PRIMARY_BASE_URL="http://localhost:8000/v1",
                  PRIMARY_API_KEY="EMPTY")
!python query_template.py
if TASK == "text":
    !python adaptive_test.py --store_path data --num_samples {NUM_SAMPLES}
elif TASK == "code":
    !python text2code_adapvetest.py --store_path data --num_samples {NUM_SAMPLES}

In [ ]:
# Bundle outputs so they show up under the notebook's Output tab.
!cd /kaggle/working/MultiCodeAttack && tar czf /kaggle/working/outputs.tgz $(ls -d response logs test code_logs code_response code_test 2>/dev/null)
!ls -la /kaggle/working